# LeetCode 3273 — Minimum Amount of Damage Dealt to Bob bằng Python

- Link: https://leetcode.com/problems/minimum-amount-of-damage-dealt-to-bob/
- Độ khó: **Hard**
- Chủ đề: **Lập lịch tham lam bằng lập luận hoán đổi** (Smith's rule: sắp theo `damage / time` giảm dần), quy đổi máu thành **số giây cần đánh** `t = ⌈health / power⌉`

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def minDamage(self, power: int, damage: List[int], health: List[int]) -> int:
```

## 1. Mục tiêu học

1. Quy đổi: kẻ địch `i` cần đúng `t_i = ⌈health_i / power⌉` giây bị đánh để chết.
2. Nhận ra **không bao giờ** nên đánh xen kẽ: đánh dứt điểm từng kẻ địch theo một thứ tự là tối ưu.
3. Với một thứ tự, tổng sát thương `= Σ damage_i · C_i` với `C_i` = thời điểm kẻ địch `i` chết ⇒ bài toán "tổng thời gian hoàn thành có trọng số" `1 || Σ w_j C_j`.
4. **Lập luận hoán đổi** hai kẻ địch kề nhau ⇒ sắp theo `damage_i / t_i` giảm dần (so sánh chéo `d_i · t_j > d_j · t_i` để tránh số thực).
5. Kiểm chứng bằng DP trạng thái đầy đủ (cho phép xen kẽ) trên dữ liệu nhỏ.

## 2. Đề bài & ví dụ

Bob có `n` kẻ địch; kẻ địch `i` gây `damage[i]` sát thương **mỗi giây** khi còn sống (`health[i] > 0`). Mỗi giây, **sau khi** các kẻ địch gây sát thương, Bob chọn một kẻ địch còn sống và gây `power` sát thương cho nó. Tìm tổng sát thương nhỏ nhất Bob phải nhận trước khi tất cả kẻ địch chết.

**Ví dụ 1:** `power = 4`, `damage = [1,2,3,4]`, `health = [4,5,6,8]` → `39`.
**Ví dụ 2:** `power = 1`, `damage = [1,1,1,1]`, `health = [1,2,3,4]` → `20`.
**Ví dụ 3:** `power = 8`, `damage = [40]`, `health = [59]` → `320`.

**Ràng buộc:** `1 ≤ power ≤ 10^4`; `1 ≤ n ≤ 10^5`; `1 ≤ damage[i], health[i] ≤ 10^4`.

In [1]:
from typing import List
from functools import cmp_to_key, lru_cache
from fractions import Fraction
from itertools import permutations

examples = [(4, [1, 2, 3, 4], [4, 5, 6, 8], 39), (1, [1, 1, 1, 1], [1, 2, 3, 4], 20), (8, [40], [59], 320)]
for p, d, h, exp in examples:
    print('power =', p, d, h, '-> kỳ vọng', exp)

power = 4 [1, 2, 3, 4] [4, 5, 6, 8] -> kỳ vọng 39
power = 1 [1, 1, 1, 1] [1, 2, 3, 4] -> kỳ vọng 20
power = 8 [40] [59] -> kỳ vọng 320


## 3. Kiến thức nền

### 3.1. Số giây cần đánh

`t_i = ⌈h_i / p⌉ = (h_i + p − 1) // p`. Ví dụ 3: `⌈59 / 8⌉ = 8` giây, sát thương `40 · 8 = 320`.

### 3.2. Tổng sát thương theo một thứ tự

Nếu đánh dứt điểm theo thứ tự `π_1, π_2, …`, kẻ `π_k` chết tại `C_{π_k} = t_{π_1} + … + t_{π_k}` và gây sát thương trong suốt `C_{π_k}` giây (mỗi giây nó còn sống, nó gây sát thương **trước** khi bị đánh).

In [2]:
def cost_of_order(power, damage, health, order):
    t, total = 0, 0
    for i in order:
        t += (health[i] + power - 1) // power
        total += damage[i] * t
    return total

print(cost_of_order(*examples[0][:3], [3, 2, 0, 1]), '(thứ tự trong lời giải đề)')

39 (thứ tự trong lời giải đề)


### 3.3. Lập luận hoán đổi

Xét hai kẻ địch kề nhau `i` rồi `j` trong thứ tự, bắt đầu ở thời điểm `T`. Chi phí riêng hai kẻ: `d_i(T + t_i) + d_j(T + t_i + t_j)`. Đổi chỗ: `d_j(T + t_j) + d_i(T + t_j + t_i)`. Hiệu (trước − sau) `= d_j t_i − d_i t_j`. Vậy `i` nên đứng trước `j` ⇔ `d_i t_j ≥ d_j t_i` ⇔ `d_i / t_i ≥ d_j / t_j`.

## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Ví dụ 1 (`power = 4`): `t = [1, 2, 2, 2]`, tỉ lệ `d / t = [1, 1, 1.5, 2]` ⇒ thứ tự `3, 2, {0, 1}` (hòa giữa 0 và 1 — thứ tự nào cũng được):

- Kẻ 3 chết ở giây 2: `4 · 2 = 8`; kẻ 2 chết ở giây 4: `3 · 4 = 12`; kẻ 0 chết ở giây 5: `1 · 5 = 5`; kẻ 1 chết ở giây 7: `2 · 7 = 14`.
- Tổng `8 + 12 + 5 + 14 = 39` ✓.

## 5. Cách 1 — Brute Force: DP trên trạng thái "số đòn còn lại" (cho phép đánh xen kẽ)

In [3]:
def brute(power, damage, health):
    hits = tuple((h + power - 1) // power for h in health)

    @lru_cache(maxsize=None)
    def f(state):
        alive = [i for i, s in enumerate(state) if s > 0]
        if not alive:
            return 0
        dmg = sum(damage[i] for i in alive)               # sát thương giây này
        best = None
        for i in alive:
            nxt = list(state); nxt[i] -= 1
            v = dmg + f(tuple(nxt))
            best = v if best is None else min(best, v)
        return best

    return f(hits)

for p, d, h, exp in examples:
    print(brute(p, d, h), '| kỳ vọng', exp)

39 | kỳ vọng 39
20 | kỳ vọng 20
320 | kỳ vọng 320


Oracle này **không giả định** đánh dứt điểm — nó cho phép mọi lịch, kể cả xen kẽ. Khớp kết quả với cách tham lam chứng minh bằng thực nghiệm rằng xen kẽ không có lợi.

## 6. Ý tưởng thuật toán / bước đột phá

1. `t_i = ⌈h_i / p⌉`.
2. Sắp chỉ số theo so sánh: `i` trước `j` nếu `d_i · t_j > d_j · t_i`.
3. Duyệt theo thứ tự, cộng dồn thời gian và `d_i · C_i`.

`O(n log n)`.

## 7. Các cách tiếp theo

### 7.1. Cách 2 — DP bitmask trên tập đã giết (thứ tự tốt nhất, `n ≤ 15`)

In [4]:
def bitmask_order(power, damage, health):
    n = len(damage)
    t = [(h + power - 1) // power for h in health]
    INF = float('inf')
    dp = [INF] * (1 << n); dp[0] = 0
    for mask in range(1 << n):
        if dp[mask] == INF:
            continue
        T = sum(t[i] for i in range(n) if mask >> i & 1)
        for i in range(n):
            if not mask >> i & 1:
                m2 = mask | 1 << i
                v = dp[mask] + damage[i] * (T + t[i])
                if v < dp[m2]:
                    dp[m2] = v
    return dp[(1 << n) - 1]

for p, d, h, exp in examples:
    print(bitmask_order(p, d, h), '| kỳ vọng', exp)

39 | kỳ vọng 39
20 | kỳ vọng 20
320 | kỳ vọng 320


### 7.2. Cách 3 — Sắp theo so sánh chéo (lời giải chính)

In [5]:
def greedy_cmp(power, damage, health):
    t = [(h + power - 1) // power for h in health]
    order = sorted(range(len(damage)), key=cmp_to_key(lambda i, j: damage[j] * t[i] - damage[i] * t[j]))
    return cost_of_order(power, damage, health, order)

for p, d, h, exp in examples:
    print(greedy_cmp(p, d, h), '| kỳ vọng', exp)

39 | kỳ vọng 39
20 | kỳ vọng 20
320 | kỳ vọng 320


Hàm so sánh trả về số âm khi `i` nên đứng trước: `d_j t_i − d_i t_j < 0` ⇔ `d_i t_j > d_j t_i`.

### 7.3. Cách 4 — Khóa sắp xếp là phân số `−d / t` (chính xác tuyệt đối)

In [6]:
def greedy_fraction(power, damage, health):
    t = [(h + power - 1) // power for h in health]
    order = sorted(range(len(damage)), key=lambda i: Fraction(-damage[i], t[i]))
    return cost_of_order(power, damage, health, order)

for p, d, h, exp in examples:
    print(greedy_fraction(p, d, h), '| kỳ vọng', exp)

39 | kỳ vọng 39
20 | kỳ vọng 20
320 | kỳ vọng 320


Khóa `Fraction` tránh sai số dấu phẩy động; chậm hơn so sánh chéo nhưng gọn.

## 8. TRACE từng bước

Trace cách 3 trên ví dụ 1.

In [7]:
def trace(power, damage, health):
    t = [(h + power - 1) // power for h in health]
    order = sorted(range(len(damage)), key=cmp_to_key(lambda i, j: damage[j] * t[i] - damage[i] * t[j]))
    print('t =', t, '| tỉ lệ d/t =', [round(d / ti, 3) for d, ti in zip(damage, t)], '| thứ tự:', order)
    T = total = 0
    for i in order:
        T += t[i]; total += damage[i] * T
        print(f'  kẻ {i}: đánh {t[i]} giây, chết ở giây {T}, gây {damage[i]} × {T} = {damage[i] * T} -> tổng {total}')
    print('đáp án:', total, '| DP trạng thái:', brute(power, damage, health))

trace(*examples[0][:3])

t = [1, 2, 2, 2] | tỉ lệ d/t = [1.0, 1.0, 1.5, 2.0] | thứ tự: [3, 2, 0, 1]
  kẻ 3: đánh 2 giây, chết ở giây 2, gây 4 × 2 = 8 -> tổng 8
  kẻ 2: đánh 2 giây, chết ở giây 4, gây 3 × 4 = 12 -> tổng 20
  kẻ 0: đánh 1 giây, chết ở giây 5, gây 1 × 5 = 5 -> tổng 25
  kẻ 1: đánh 2 giây, chết ở giây 7, gây 2 × 7 = 14 -> tổng 39
đáp án: 39 | DP trạng thái: 39


## 9. Code nộp LeetCode

In [8]:
class Solution:
    def minDamage(self, power: int, damage: List[int], health: List[int]) -> int:
        t = [(h + power - 1) // power for h in health]
        order = sorted(range(len(damage)), key=cmp_to_key(lambda i, j: damage[j] * t[i] - damage[i] * t[j]))
        elapsed = 0
        total = 0
        for i in order:
            elapsed += t[i]
            total += damage[i] * elapsed
        return total

for p, d, h, exp in examples:
    got = Solution().minDamage(p, d, h)
    print(got, 'OK' if got == exp else 'SAI')

39 OK
20 OK
320 OK


## 10. Vì sao thuật toán đúng

1. **Không xen kẽ:** giả sử trong một lịch tối ưu, kẻ `i` chết trước kẻ `j` nhưng có một đòn đánh `j` xảy ra trước đòn cuối của `i`. Dời đòn đó ra sau đòn cuối của `i` (giữ thứ tự các đòn khác): `i` chết sớm hơn ít nhất 1 giây, `j` chết đúng thời điểm cũ, các kẻ khác không đổi ⇒ tổng không tăng. Lặp lại ⇒ có lịch tối ưu đánh dứt điểm.
2. **Chi phí theo thứ tự:** mục 3.2.
3. **Hoán đổi kề:** mục 3.3 — nếu tồn tại cặp kề `i, j` với `d_i t_j < d_j t_i`, đổi chỗ làm giảm tổng nghiêm ngặt. Do đó thứ tự tối ưu không có "nghịch thế" theo tỉ lệ ⇒ sắp theo `d / t` giảm dần (các phần tử hòa đổi chỗ tùy ý không ảnh hưởng tổng).

## 11. Phân tích Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. DP trạng thái | `O(∏ (t_i + 1) · n)` | như trên |
| 2. DP bitmask thứ tự | `O(2^n · n²)` | `O(2^n)` |
| 3. Sắp so sánh chéo | `O(n log n)` | `O(n)` |
| 4. Sắp khóa phân số | `O(n log n)` (hằng số lớn) | `O(n)` |

## 12. Bộ test

### 12.1. Ví dụ đề + ca biên

In [9]:
impls = {'bitmask_order': bitmask_order, 'greedy_cmp': greedy_cmp, 'greedy_fraction': greedy_fraction,
         'submit': lambda p, d, h: Solution().minDamage(p, d, h)}
edge = [(1, [1], [1], 1), (10, [5, 5], [1, 1], 15), (3, [2, 1], [3, 9], 6), (2, [1, 100], [100, 2], 151)]
for p, d, h, exp in examples + edge:
    assert brute(p, d, h) == exp, (p, d, h, brute(p, d, h))
    for name, f in impls.items():
        assert f(p, d, h) == exp, (name, p, d, h)
print('Ví dụ đề và ca biên: tất cả đúng.')

Ví dụ đề và ca biên: tất cả đúng.


### 12.2. Đối chiếu ngẫu nhiên với DP trạng thái (cho phép xen kẽ)

In [10]:
import random
rng = random.Random(3273)
cnt = 0
for _ in range(400):
    n = rng.randint(1, 4)
    p = rng.randint(1, 5)
    d = [rng.randint(1, 9) for _ in range(n)]
    h = [rng.randint(1, 12) for _ in range(n)]
    exp = brute(p, d, h)
    for name, f in impls.items():
        assert f(p, d, h) == exp, (name, p, d, h)
    cnt += 1
print('Khớp DP trạng thái trên', cnt, 'bộ test — xác nhận đánh xen kẽ không có lợi.')

Khớp DP trạng thái trên 400 bộ test — xác nhận đánh xen kẽ không có lợi.


### 12.3. Ca lớn `n = 10^5`

In [11]:
import time
N = 10 ** 5
big = {
    'ngẫu nhiên': (rng.randint(1, 10 ** 4), [rng.randint(1, 10 ** 4) for _ in range(N)], [rng.randint(1, 10 ** 4) for _ in range(N)]),
    'power = 1': (1, [rng.randint(1, 10 ** 4) for _ in range(N)], [rng.randint(1, 10 ** 4) for _ in range(N)]),
    'đồng nhất': (7, [5] * N, [20] * N),
}
for name, (p, d, h) in big.items():
    t = time.time(); a = Solution().minDamage(p, d, h); t1 = time.time() - t
    t = time.time(); b = greedy_fraction(p, d, h); t2 = time.time() - t
    assert a == b
    print(f"{name:12s}: {a:22d} | nộp {t1:.2f}s | phân số {t2:.2f}s")
tt = (20 + 6) // 7
assert Solution().minDamage(7, [5] * N, [20] * N) == 5 * tt * N * (N + 1) // 2
print('đồng nhất: tổng = d·t·N(N+1)/2 — đúng.')

ngẫu nhiên  :         16641174307560 | nộp 0.32s | phân số 0.74s


power = 1   :      64819315854452904 | nộp 0.32s | phân số 0.65s
đồng nhất   :            75000750000 | nộp 0.05s | phân số 0.15s


đồng nhất: tổng = d·t·N(N+1)/2 — đúng.


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Sắp theo `damage` giảm dần (bỏ qua thời gian giết)

In [12]:
def wrong_by_damage(power, damage, health):
    order = sorted(range(len(damage)), key=lambda i: -damage[i])
    return cost_of_order(power, damage, health, order)

bad = (1, [3, 2], [10, 1])
print('sai :', wrong_by_damage(*bad), '| đúng:', Solution().minDamage(*bad))

sai : 52 | đúng: 35


**Vì sao sai:** kẻ `3`/giây cần 10 giây, kẻ `2`/giây chỉ cần 1 giây. Giết kẻ mạnh trước: `3·10 + 2·11 = 52`; giết kẻ yếu (nhưng nhanh chết) trước: `2·1 + 3·11 = 35`. Phải so **tỉ lệ** `d / t` (`0.3` so với `2`).

### Lỗi 2 — Sắp theo thời gian giết tăng dần (bỏ qua sát thương)

In [13]:
def wrong_by_time(power, damage, health):
    order = sorted(range(len(damage)), key=lambda i: (health[i] + power - 1) // power)
    return cost_of_order(power, damage, health, order)

print('sai :', wrong_by_time(*examples[0][:3]), '| đúng:', Solution().minDamage(*examples[0][:3]))

sai : 50 | đúng: 39


**Vì sao sai:** kẻ 0 chết nhanh nhất (1 giây) nhưng chỉ gây 1 sát thương/giây — đáng ưu tiên kẻ 3 (4/giây, 2 giây) hơn.

### Lỗi 3 — Dùng `health` thay vì số giây `⌈health / power⌉`

In [14]:
def wrong_raw_health(power, damage, health):
    order = sorted(range(len(damage)), key=cmp_to_key(lambda i, j: damage[j] * health[i] - damage[i] * health[j]))
    return cost_of_order(power, damage, health, order)

**Vì sao sai:** thứ tự phụ thuộc vào **số giây** bị đánh; làm tròn lên `⌈h / p⌉` có thể đảo thứ tự tỉ lệ so với máu thô. Ô dưới tìm tự động một phản ví dụ.

In [15]:
found = None
r_ = random.Random(2)
for _ in range(20000):
    p = r_.randint(2, 6); d = [r_.randint(1, 9) for _ in range(2)]; h = [r_.randint(1, 15) for _ in range(2)]
    if wrong_raw_health(p, d, h) != Solution().minDamage(p, d, h):
        found = (p, d, h); break
print('phản ví dụ:', found, '-> sai:', wrong_raw_health(*found), '| đúng:', Solution().minDamage(*found))

phản ví dụ: (5, [2, 5], [2, 10]) -> sai: 17 | đúng: 16


### Lỗi 4 — Làm tròn xuống `health // power`

In [16]:
def wrong_floor(power, damage, health):
    t = [max(1, h // power) for h in health]                  # SAI: phải làm tròn lên
    order = sorted(range(len(damage)), key=cmp_to_key(lambda i, j: damage[j] * t[i] - damage[i] * t[j]))
    el = tot = 0
    for i in order:
        el += t[i]; tot += damage[i] * el
    return tot

print('sai :', wrong_floor(*examples[2][:3]), '| đúng:', Solution().minDamage(*examples[2][:3]))

sai : 280 | đúng: 320


**Vì sao sai:** máu `59`, `power = 8`: 7 đòn chỉ gây 56 < 59 — cần đòn thứ 8.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Mỗi việc tốn thời gian `t_i`, chịu phạt `w_i` mỗi đơn vị thời gian chưa xong" ⇒ **Smith's rule**: sắp `w / t` giảm dần.
- Chứng minh tham lam bằng **hoán đổi hai phần tử kề**.
- So sánh tỉ lệ bằng **nhân chéo** (tránh số thực), dùng `cmp_to_key`.

## 15. Biến thể của bài

1. Bob đánh **trước** khi kẻ địch gây sát thương ⇒ kẻ chết ở giây `C` chỉ gây `C − 1` giây.
2. Có thể đánh **nhiều** kẻ một lúc (chia `power`) — không còn đơn giản.
3. Kẻ địch có thời điểm **xuất hiện** khác nhau ⇒ NP-khó tổng quát (`1 | r_j | Σ w_j C_j`).
4. Tối thiểu **sát thương lớn nhất trong một giây** — bài khác (luôn bằng tổng ban đầu).

In [17]:
# Biến thể 1: Bob đánh trước
def attack_first(power, damage, health):
    t = [(h + power - 1) // power for h in health]
    order = sorted(range(len(damage)), key=cmp_to_key(lambda i, j: damage[j] * t[i] - damage[i] * t[j]))
    el = tot = 0
    for i in order:
        el += t[i]; tot += damage[i] * (el - 1)
    return tot

def attack_first_brute(power, damage, health):
    n = len(damage); best = None
    for perm in permutations(range(n)):
        el = tot = 0
        for i in perm:
            el += (health[i] + power - 1) // power; tot += damage[i] * (el - 1)
        best = tot if best is None else min(best, tot)
    return best

rng = random.Random(15)
for _ in range(300):
    n = rng.randint(1, 5); p = rng.randint(1, 5)
    d = [rng.randint(1, 9) for _ in range(n)]; h = [rng.randint(1, 15) for _ in range(n)]
    assert attack_first(p, d, h) == attack_first_brute(p, d, h) == Solution().minDamage(p, d, h) - sum(d)
print('Biến thể "Bob đánh trước": đáp án = đáp án gốc − Σ damage (khớp vét cạn).')

Biến thể "Bob đánh trước": đáp án = đáp án gốc − Σ damage (khớp vét cạn).


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Ghi chú |
|---|---|---|---|
| 1 | DP trạng thái (xen kẽ được) | mũ | Oracle mạnh |
| 2 | DP bitmask thứ tự | `O(2^n n²)` | Oracle thứ tự |
| 3 | Smith's rule, nhân chéo | `O(n log n)` | **Nộp bài** |
| 4 | Smith's rule, phân số | `O(n log n)` | Chính xác, chậm hơn |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** `power = 5`, `damage = [2]`, `health = [11]`.

**Bài 2 (tính tay):** `power = 2`, `damage = [3, 1]`, `health = [4, 1]`.

**Bài 3 (điền TODO):** trả về **thứ tự** giết tối ưu (hòa thì chỉ số nhỏ trước).

**Bài 4 (tự viết lại):** đếm số thứ tự tối ưu khác nhau (nhóm hòa tỉ lệ hoán vị tự do).

**Bài 5 (mini challenge):** chứng minh bằng code rằng tổng phạt **không đổi** khi hoán vị trong nhóm cùng tỉ lệ.

In [18]:
def best_order_todo(power, damage, health):
    # TODO
    raise NotImplementedError

def count_optimal_orders_todo(power, damage, health):
    # TODO
    raise NotImplementedError

def ties_invariant_todo(power, damage, health):
    # TODO
    raise NotImplementedError

print('Đã định nghĩa các hàm TODO (không gọi).')

Đã định nghĩa các hàm TODO (không gọi).


## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) `t = 3`. (2) `2 · 3`. (3) `6`.

**Bài 2.** (1) `t = [2, 1]`, tỉ lệ `1.5` và `1`. (2) Giết kẻ 0 trước: `3·2 + 1·3`. (3) `9`.

**Bài 3.** (1) Khóa `(Fraction(−d, t), i)`. (2) Sắp. (3) Trả danh sách.

**Bài 4.** (1) Gom nhóm theo tỉ lệ (phân số tối giản). (2) Tích các giai thừa kích thước nhóm. (3) Đối chiếu đếm hoán vị tối ưu.

**Bài 5.** (1) Hai kẻ hòa tỉ lệ: hiệu hoán đổi `d_j t_i − d_i t_j = 0`. (2) Mọi hoán vị trong nhóm là tích các hoán đổi kề. (3) Kiểm bằng mọi hoán vị nhóm.

## 19. Đáp án tham khảo cho bài tập

In [19]:
# Bài 1, 2, 3
assert Solution().minDamage(5, [2], [11]) == 6
assert Solution().minDamage(2, [3, 1], [4, 1]) == 9 == brute(2, [3, 1], [4, 1])

def best_order(power, damage, health):
    t = [(h + power - 1) // power for h in health]
    return sorted(range(len(damage)), key=lambda i: (Fraction(-damage[i], t[i]), i))

o = best_order(*examples[0][:3])
assert cost_of_order(*examples[0][:3], o) == 39
print('Bài 1: 6 | Bài 2: 9 | Bài 3: thứ tự', o)

Bài 1: 6 | Bài 2: 9 | Bài 3: thứ tự [3, 2, 0, 1]


In [20]:
# Bài 4, 5
from math import factorial
from collections import Counter
def count_optimal_orders(power, damage, health):
    t = [(h + power - 1) // power for h in health]
    groups = Counter(Fraction(d, ti) for d, ti in zip(damage, t))
    res = 1
    for c in groups.values():
        res *= factorial(c)
    return res

def ties_invariant(power, damage, health):
    best = Solution().minDamage(power, damage, health)
    return all(cost_of_order(power, damage, health, perm) == best
               for perm in permutations(range(len(damage)))
               if cost_of_order(power, damage, health, perm) <= best)

rng = random.Random(4)
for _ in range(200):
    n = rng.randint(1, 5); p = rng.randint(1, 4)
    d = [rng.randint(1, 4) for _ in range(n)]; h = [rng.randint(1, 8) for _ in range(n)]
    best = Solution().minDamage(p, d, h)
    cnt_opt = sum(1 for perm in permutations(range(n)) if cost_of_order(p, d, h, perm) == best)
    assert cnt_opt == count_optimal_orders(p, d, h), (p, d, h)
    assert ties_invariant(p, d, h)
print('Bài 4: số thứ tự tối ưu = tích giai thừa các nhóm hòa (khớp vét cạn). Bài 5 đúng.')

Bài 4: số thứ tự tối ưu = tích giai thừa các nhóm hòa (khớp vét cạn). Bài 5 đúng.


Ghi chú bài 4: kết quả khớp vét cạn cho thấy **mọi** thứ tự tối ưu đều tôn trọng thứ tự tỉ lệ (không có thứ tự tối ưu "lạ" nào khác) — đúng với lập luận hoán đổi chặt (đổi chỗ cặp nghịch thế làm giảm **nghiêm ngặt**).

## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
t = [(h + p - 1) // p for h in health]
order = sorted(range(n), key=cmp_to_key(lambda i, j: d[j]*t[i] - d[i]*t[j]))
elapsed = total = 0
for i in order: elapsed += t[i]; total += d[i] * elapsed
```

**Checklist:**

- [ ] `t_i` làm tròn **lên**?
- [ ] Sắp theo **tỉ lệ** `d / t`, không chỉ `d` hay `t`?
- [ ] So sánh chéo bằng số nguyên?
- [ ] Sát thương tính cả giây kẻ địch chết (đánh sau khi bị gây sát thương)?
- [ ] Không cần xét đánh xen kẽ (đã chứng minh)?